---
id: "readme"
template: "markdown"
name: "readme"
position:
  x: 0
  y: -40
dimensions:
  width: 900
  height: 360
config:
  md: |-
    # m_TOP_PRODUCTS_PER_STORE (converted from Informatica PowerCenter)

    | Informatica | Lakeflow Designer |
    |---|---|
    | SQ_ORDERS (filter STATUS/CHANNEL) | `src_orders` + `sq_orders_filter` |
    | JNR_ORDERS_LINES | `jnr_orders_lines` |
    | LKP_PRODUCTS (connected) | `lkp_products` |
    | EXP_NET_AMOUNT | `exp_net_amount` |
    | AGG_STORE_PRODUCT | `agg_store_product` + `agg_round` |
    | RNK_TOP3_PER_STORE (Top 3, ties share a rank) | `rnk_rank` (SQL RANK) + `rnk_top3` (Filter) |
    | :LKP.LKP_STORE / :LKP.LKP_STORE_REGION (unconnected) | `lkp_stores` (one left join returns both ports) + `exp_store_attrs` |
    | SRT_REPORT + SEQ_REPORT_ROW (NEXTVAL, start 1000) | `seq_report_row` (ROW_NUMBER over the sorter keys + 999) |

    Sequence values follow the Sorter order (STORE_ID, RANKINDEX, PRODUCT_ID), so they are deterministic here.
---

In [0]:
"""
id: "src_orders"
template: "source"
templateVersion: "1.0.0"
name: "src_orders"
position:
  x: 0
  y: 360
description:
  text: "SQ_ORDERS source."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_orders"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_orders"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_orders.data"] = out["data"]


In [0]:
"""
id: "sq_orders_filter"
template: "filter"
templateVersion: "1.0.0"
name: "sq_orders_filter"
position:
  x: 300
  y: 360
description:
  text: "SQ_ORDERS Source Filter."
  hash: ""
previewMode: "1000"
config:
  condition: "status = 'COMPLETED' AND channel = 'STORE'"
input:
  - node: "src_orders"
    input_port: "data"
    output_port: "data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df}

    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "status = 'COMPLETED' AND channel = 'STORE'"
}
inputs = {
    "data": ctx["src_orders.data"]
}
out = run(config, inputs, spark)
ctx["sq_orders_filter.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: "src_order_lines"
template: "source"
templateVersion: "1.0.0"
name: "src_order_lines"
position:
  x: 0
  y: 515
description:
  text: "SQ_ORDER_LINES source."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_order_lines"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_order_lines"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_order_lines.data"] = out["data"]


In [0]:
"""
id: "src_products"
template: "source"
templateVersion: "1.0.0"
name: "src_products"
position:
  x: 0
  y: 670
description:
  text: "LKP_PRODUCTS lookup source."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_products"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_products"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_products.data"] = out["data"]


In [0]:
"""
id: "src_stores"
template: "source"
templateVersion: "1.0.0"
name: "src_stores"
position:
  x: 0
  y: 825
description:
  text: "LKP_STORE / LKP_STORE_REGION lookup source."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_stores"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_stores"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_stores.data"] = out["data"]


In [0]:
"""
id: "jnr_orders_lines"
template: "join"
templateVersion: "1.0.0"
name: "jnr_orders_lines"
position:
  x: 600
  y: 360
description:
  text: "JNR_ORDERS_LINES: Normal Join."
  hash: ""
previewMode: "1000"
config:
  join_type: "inner"
  join_conditions: "left.`order_id` = right.`order_id`"
  expressions:
    - "left.`store_id` AS `store_id`"
    - "right.`product_id` AS `product_id`"
    - "right.`quantity` AS `quantity`"
    - "right.`unit_price` AS `unit_price`"
    - "right.`discount_pct` AS `discount_pct`"
input:
  - node: "sq_orders_filter"
    input_port: "left"
    output_port: "filtered_data"
  - node: "src_order_lines"
    input_port: "right"
    output_port: "data"
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    # Alias DataFrames with fixed names so that table-qualified column
    # references in join_conditions and expressions always resolve.
    # The TS serializer emits "left.col" / "right.col" regardless of the
    # actual table names, so these aliases must match exactly.
    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    # join_condition is a plain SQL expression string serialized by the TS
    # side (same pattern as filter's condition field).
    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    # Apply optional SELECT expressions
    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "inner",
    "join_conditions": "left.`order_id` = right.`order_id`",
    "expressions": [
        "left.`store_id` AS `store_id`",
        "right.`product_id` AS `product_id`",
        "right.`quantity` AS `quantity`",
        "right.`unit_price` AS `unit_price`",
        "right.`discount_pct` AS `discount_pct`"
    ]
}
inputs = {
    "left": ctx["sq_orders_filter.filtered_data"],
    "right": ctx["src_order_lines.data"]
}
out = run(config, inputs, spark)
ctx["jnr_orders_lines.joined_data"] = out["joined_data"]


In [0]:
"""
id: "lkp_products"
template: "join"
templateVersion: "1.0.0"
name: "lkp_products"
position:
  x: 900
  y: 360
description:
  text: "LKP_PRODUCTS (connected)."
  hash: ""
previewMode: "1000"
config:
  join_type: "left"
  join_conditions: "left.`product_id` = right.`product_id`"
  expressions:
    - "left.*"
    - "right.`product_name` AS `product_name`"
    - "right.`category` AS `category`"
    - "right.`list_price` AS `list_price`"
input:
  - node: "jnr_orders_lines"
    input_port: "left"
    output_port: "joined_data"
  - node: "src_products"
    input_port: "right"
    output_port: "data"
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    # Alias DataFrames with fixed names so that table-qualified column
    # references in join_conditions and expressions always resolve.
    # The TS serializer emits "left.col" / "right.col" regardless of the
    # actual table names, so these aliases must match exactly.
    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    # join_condition is a plain SQL expression string serialized by the TS
    # side (same pattern as filter's condition field).
    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    # Apply optional SELECT expressions
    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.`product_id` = right.`product_id`",
    "expressions": [
        "left.*",
        "right.`product_name` AS `product_name`",
        "right.`category` AS `category`",
        "right.`list_price` AS `list_price`"
    ]
}
inputs = {
    "left": ctx["jnr_orders_lines.joined_data"],
    "right": ctx["src_products.data"]
}
out = run(config, inputs, spark)
ctx["lkp_products.joined_data"] = out["joined_data"]


In [0]:
"""
id: "exp_net_amount"
template: "transform"
templateVersion: "1.0.0"
name: "exp_net_amount"
position:
  x: 1200
  y: 360
description:
  text: "EXP_NET_AMOUNT."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "store_id"
    - "product_id"
    - "product_name"
    - "category"
    - "quantity"
    - "ROUND(quantity * COALESCE(unit_price, list_price) * (1 - discount_pct), 2) AS net_amount"
input:
  - node: "lkp_products"
    input_port: "data"
    output_port: "joined_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "store_id",
        "product_id",
        "product_name",
        "category",
        "quantity",
        "ROUND(quantity * COALESCE(unit_price, list_price) * (1 - discount_pct), 2) AS net_amount"
    ]
}
inputs = {
    "data": ctx["lkp_products.joined_data"]
}
out = run(config, inputs, spark)
ctx["exp_net_amount.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "agg_store_product"
template: "aggregate"
templateVersion: "1.0.0"
name: "agg_store_product"
position:
  x: 1500
  y: 360
description:
  text: "AGG_STORE_PRODUCT: group by STORE_ID, PRODUCT_ID (PRODUCT_NAME/CATEGORY are functionally dependent, so grouped instead of last-row passthrough)."
  hash: ""
previewMode: "1000"
config:
  group_bys:
    - expr: "store_id"
      type: "expr"
    - expr: "product_id"
      type: "expr"
    - expr: "product_name"
      type: "expr"
    - expr: "category"
      type: "expr"
  aggregations:
    - columnExpr:
        expr: "net_amount"
        type: "expr"
      fn: "SUM"
      alias: "revenue_raw"
    - columnExpr:
        expr: "quantity"
        type: "expr"
      fn: "SUM"
      alias: "units"
input:
  - node: "exp_net_amount"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
import math
from typing import Dict, Any
import pyspark.sql.functions as F

# Default percentile (median). Used as a fallback when a PERCENTILE aggregation
# is missing or has a non-numeric / non-finite percentage. Mirrors
# DEFAULT_PERCENTILE_VALUE in AggregateNodeViewModel.ts so SQL and Python
# pipelines agree on the default.
DEFAULT_PERCENTILE = 0.5

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs.get("data")
    group_bys = config.get("group_bys", [])
    aggregations = config.get("aggregations", [])

    group_by_set = set(e for gb in group_bys if (e := gb.get("expr", "")))

    agg_exprs = []
    for agg_def in aggregations:
        col_expr = agg_def.get("columnExpr", {})
        raw_expr = col_expr.get("expr", "")
        fn = agg_def.get("fn", "-")
        alias = agg_def.get("alias")

        # Skip pass-through columns that duplicate a group-by column;
        # groupBy() already includes them in the output.
        if (fn == "-" or fn == "_") and not alias and raw_expr in group_by_set:
            continue

        fn_map = {
            "SUM": F.sum,
            "AVG": F.avg,
            "COUNT": F.count,
            "MIN": F.min,
            "MAX": F.max,
            "MEAN": F.mean,
            "MEDIAN": F.median,
            "STDDEV": F.stddev,
            "VARIANCE": F.variance,
        }

        agg_fn = fn_map.get(fn)
        if agg_fn:
            col = agg_fn(raw_expr)
        elif fn == "-" or fn == "_":
            col = F.col(raw_expr)
        elif fn == "PERCENTILE":
            # Spark PERCENTILE requires (col, percentage [, frequency]) with percentage
            # in [0, 1]; emitting PERCENTILE(col) triggers WRONG_NUM_ARGS at analysis
            # time. For finite numeric inputs we clamp to [0, 1] (mirroring the V1/V2
            # input clamps) so out-of-range values like -0.1 or 1.5 — typos or stale
            # frontend state — become a valid boundary instead of silently overriding
            # the user's intent with the median. Non-numeric, non-finite, boolean, or
            # missing values fall back to DEFAULT_PERCENTILE.
            raw_pct = agg_def.get("percentage")
            if (
                isinstance(raw_pct, (int, float))
                and not isinstance(raw_pct, bool)
                and math.isfinite(raw_pct)
            ):
                pct = max(0.0, min(1.0, float(raw_pct)))
            else:
                pct = DEFAULT_PERCENTILE
            col = F.expr(f"PERCENTILE({raw_expr}, {pct})")
        else:
            col = F.expr(f"{fn}({raw_expr})")

        if alias:
            col = col.alias(alias)

        agg_exprs.append(col)

    group_cols = [
        gb.get("expr", "") for gb in group_bys if gb.get("expr", "")
    ]

    if not agg_exprs:
        if group_cols:
            result = df.select(*group_cols).distinct()
            return {"aggregated_data": result}
        return {"aggregated_data": df}

    if group_cols:
        result = df.groupBy(*group_cols).agg(*agg_exprs)
    else:
        result = df.agg(*agg_exprs)

    return {"aggregated_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "group_bys": [
        {
            "expr": "store_id",
            "type": "expr"
        },
        {
            "expr": "product_id",
            "type": "expr"
        },
        {
            "expr": "product_name",
            "type": "expr"
        },
        {
            "expr": "category",
            "type": "expr"
        }
    ],
    "aggregations": [
        {
            "columnExpr": {
                "expr": "net_amount",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "revenue_raw"
        },
        {
            "columnExpr": {
                "expr": "quantity",
                "type": "expr"
            },
            "fn": "SUM",
            "alias": "units"
        }
    ]
}
inputs = {
    "data": ctx["exp_net_amount.transformed_data"]
}
out = run(config, inputs, spark)
ctx["agg_store_product.aggregated_data"] = out["aggregated_data"]


In [0]:
"""
id: "agg_round"
template: "transform"
templateVersion: "1.0.0"
name: "agg_round"
position:
  x: 1800
  y: 360
description:
  text: "REVENUE = ROUND(SUM(..),2)."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "store_id"
    - "product_id"
    - "product_name"
    - "category"
    - "units"
    - "ROUND(revenue_raw, 2) AS revenue"
input:
  - node: "agg_store_product"
    input_port: "data"
    output_port: "aggregated_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "store_id",
        "product_id",
        "product_name",
        "category",
        "units",
        "ROUND(revenue_raw, 2) AS revenue"
    ]
}
inputs = {
    "data": ctx["agg_store_product.aggregated_data"]
}
out = run(config, inputs, spark)
ctx["agg_round.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "rnk_rank"
template: "sql"
templateVersion: "1.0.0"
name: "rnk_rank"
position:
  x: 2100
  y: 360
description:
  text: "RNK_TOP3_PER_STORE: RANKINDEX = RANK() (ties share a rank, next value skipped)."
  hash: ""
previewMode: "1000"
config:
  query: "SELECT *, RANK() OVER (PARTITION BY store_id ORDER BY revenue DESC) AS rank_in_store FROM agg_round"
input:
  - node: "agg_round"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT *, RANK() OVER (PARTITION BY store_id ORDER BY revenue DESC) AS rank_in_store FROM agg_round"
}
inputs = {
    "data": [
        ctx["agg_round.transformed_data"]
    ],
    "data__sources": [
        {
            "node": "agg_round",
            "output_port": "transformed_data",
            "name": "agg_round",
            "df_name": "agg_round"
        }
    ]
}
out = run(config, inputs, spark)
ctx["rnk_rank.result"] = out["result"]


In [0]:
"""
id: "rnk_top3"
template: "filter"
templateVersion: "1.0.0"
name: "rnk_top3"
position:
  x: 2400
  y: 360
description:
  text: "RNK_TOP3_PER_STORE: Number Of Ranks = 3, Top."
  hash: ""
previewMode: "1000"
config:
  condition: "rank_in_store <= 3"
input:
  - node: "rnk_rank"
    input_port: "data"
    output_port: "result"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df}

    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "rank_in_store <= 3"
}
inputs = {
    "data": ctx["rnk_rank.result"]
}
out = run(config, inputs, spark)
ctx["rnk_top3.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: "lkp_stores"
template: "join"
templateVersion: "1.0.0"
name: "lkp_stores"
position:
  x: 2700
  y: 360
description:
  text: "Unconnected lookups :LKP.LKP_STORE / :LKP.LKP_STORE_REGION."
  hash: ""
previewMode: "1000"
config:
  join_type: "left"
  join_conditions: "left.`store_id` = right.`store_id`"
  expressions:
    - "left.*"
    - "right.`store_name` AS `lkp_store_name`"
    - "right.`region` AS `lkp_region`"
input:
  - node: "rnk_top3"
    input_port: "left"
    output_port: "filtered_data"
  - node: "src_stores"
    input_port: "right"
    output_port: "data"
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    # Alias DataFrames with fixed names so that table-qualified column
    # references in join_conditions and expressions always resolve.
    # The TS serializer emits "left.col" / "right.col" regardless of the
    # actual table names, so these aliases must match exactly.
    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    # join_condition is a plain SQL expression string serialized by the TS
    # side (same pattern as filter's condition field).
    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    # Apply optional SELECT expressions
    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.`store_id` = right.`store_id`",
    "expressions": [
        "left.*",
        "right.`store_name` AS `lkp_store_name`",
        "right.`region` AS `lkp_region`"
    ]
}
inputs = {
    "left": ctx["rnk_top3.filtered_data"],
    "right": ctx["src_stores.data"]
}
out = run(config, inputs, spark)
ctx["lkp_stores.joined_data"] = out["joined_data"]


In [0]:
"""
id: "exp_store_attrs"
template: "transform"
templateVersion: "1.0.0"
name: "exp_store_attrs"
position:
  x: 3000
  y: 360
description:
  text: "EXP_STORE_ATTRS: IIF(ISNULL(:LKP...), 'UNKNOWN', ...)."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "*"
    - "COALESCE(lkp_store_name, 'UNKNOWN') AS store_name"
    - "lkp_region AS region"
input:
  - node: "lkp_stores"
    input_port: "data"
    output_port: "joined_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "*",
        "COALESCE(lkp_store_name, 'UNKNOWN') AS store_name",
        "lkp_region AS region"
    ]
}
inputs = {
    "data": ctx["lkp_stores.joined_data"]
}
out = run(config, inputs, spark)
ctx["exp_store_attrs.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "seq_report_row"
template: "sql"
templateVersion: "1.0.0"
name: "seq_report_row"
position:
  x: 3300
  y: 360
description:
  text: "SRT_REPORT + SEQ_REPORT_ROW: NEXTVAL (Start Value 1000) in sorter order."
  hash: ""
previewMode: "1000"
config:
  query: |-
    SELECT 999 + ROW_NUMBER() OVER (ORDER BY store_id, rank_in_store, product_id) AS report_row_id, store_id, store_name, region, rank_in_store,
           product_id, product_name, category, units, revenue
    FROM exp_store_attrs
input:
  - node: "exp_store_attrs"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT 999 + ROW_NUMBER() OVER (ORDER BY store_id, rank_in_store, product_id) AS report_row_id, store_id, store_name, region, rank_in_store,\n       product_id, product_name, category, units, revenue\nFROM exp_store_attrs"
}
inputs = {
    "data": [
        ctx["exp_store_attrs.transformed_data"]
    ],
    "data__sources": [
        {
            "node": "exp_store_attrs",
            "output_port": "transformed_data",
            "name": "exp_store_attrs",
            "df_name": "exp_store_attrs"
        }
    ]
}
out = run(config, inputs, spark)
ctx["seq_report_row.result"] = out["result"]


In [0]:
"""
id: "out_top_products_store"
template: "output"
templateVersion: "1.0.0"
name: "out_top_products_store"
position:
  x: 3600
  y: 360
description:
  text: "T_TOP_PRODUCTS_STORE (truncate + load)."
  hash: ""
previewMode: "full"
config:
  catalog: "daphnedemos_catalog"
  schema: "migration_skill_tests"
  table_name: "infa_04_top_products_store"
input:
  - node: "seq_report_row"
    input_port: "data"
    output_port: "result"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "daphnedemos_catalog",
    "schema": "migration_skill_tests",
    "table_name": "infa_04_top_products_store"
}
inputs = {
    "data": ctx["seq_report_row.result"]
}
out = run(config, inputs, spark)
